
# ESG 단계3 연관분석 실행 노트북

이 노트북은 단계 I 전처리 JSONL을 입력으로 받아 단계 III 연관분석을 수행한다.

## 설계 기준
- 단계 III는 **추가 전처리 수정 단계가 아니라**, 현재 산출물을 바탕으로 **ESG-only 의미망**과 **open-vocab 보조표**를 만드는 단계로 둔다.
- 공출현 단위는 **문장 단위**를 기본으로 한다.
- 연관 강도 지표는 **Ochiai 계수**를 사용한다.
- 결과는 **raw / concept**를 분리 저장한다.
- **Track 간 결과를 합쳐 단일 순위표로 해석하지 않는다.**
- open-vocab 결과는 **맥락 보완 및 false positive 점검용** 보조표로만 사용한다.

## 입력
- `ESG_preprocessed_all.jsonl`
- `ESG_preprocessed_main.jsonl`

## 출력
- `단계3_연관분석/연관분석결과_all.xlsx`
- `단계3_연관분석/연관분석결과_main.xlsx`
- QA용 csv / json


## Cell 1. 라이브러리 임포트

In [1]:

import json
import math
import re
import ast
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)

try:
    import kss
    HAS_KSS = True
except Exception:
    HAS_KSS = False


## Cell 2. 경로 설정

In [2]:

# =========================
# 0. 경로 설정
# =========================
INPUT_ALL = r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_all.jsonl"
INPUT_MAIN = r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_main.jsonl"
OUTPUT_DIR = Path(r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계3_연관분석")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_CONFIG = {
    "all": INPUT_ALL,
    "main": INPUT_MAIN,
}

# =========================
# 1. 단계3 기본 필터
# =========================
MIN_ANCHOR_SENT_N = 10   # anchor_doc_n ≥ 10에 대응하는 문장 단위 최소치
MIN_CO_SENT_N = 2        # co_doc_n ≥ 2
MIN_UNIV_N = 2           # 재현 대학 수 ≥ 2
TOPN_OPEN_VOCAB = 50     # anchor별 open-vocab 상위 몇 개 저장할지
TOPN_SENTENCE_SAMPLE = 10 # edge별 예시 문장 수


## Cell 3. ESG 키워드

In [3]:

# =========================
# 2. ESG 사전 / concept mapping
# =========================
# 주의:
# - 단계3에서는 raw / concept를 분리 저장한다.
# - concept는 raw를 대체하지 않는 보조 결과다.
# - 명시적 번역 대응어만 병합한다.

ESG_CONCEPT_SPECS = {
    # ESG meta
    "ESG": {"category": "ESG", "aliases": ["ESG"]},
    "지속가능경영": {"category": "ESG", "aliases": ["지속가능경영", "sustainability management"]},
    "지속가능성보고서": {"category": "ESG", "aliases": ["지속가능성 보고서", "sustainability report"]},
    "비재무정보": {"category": "ESG", "aliases": ["비재무정보", "non-financial data", "비재무"]},
    "통합보고서": {"category": "ESG", "aliases": ["통합보고서", "integrated report"]},
    "임팩트투자": {"category": "ESG", "aliases": ["임팩트 투자", "impact investing"]},
    "책임투자": {"category": "ESG", "aliases": ["책임투자", "responsible investment", "ri"]},
    "스튜어드십코드": {"category": "ESG", "aliases": ["스튜어드십 코드", "stewardship code"]},
    "TCFD": {"category": "ESG", "aliases": ["TCFD"]},
    "SASB": {"category": "ESG", "aliases": ["SASB"]},
    "GRI": {"category": "ESG", "aliases": ["GRI"]},
    "ISSB": {"category": "ESG", "aliases": ["ISSB"]},
    "ESG지표": {"category": "ESG", "aliases": ["ESG KPI", "ESG지표", "ESG KPI"]},
    "지속가능금융": {"category": "ESG", "aliases": ["지속가능금융", "sustainable finance"]},
    "녹색채권": {"category": "ESG", "aliases": ["녹색채권", "green bond"]},
    "사회적채권": {"category": "ESG", "aliases": ["사회적 채권", "social bond"]},

    # E
    "지속가능": {"category": "E", "aliases": ["지속가능", "sustainable"]},
    "지속가능성": {"category": "E", "aliases": ["지속가능성", "sustainability"]},
    "그린": {"category": "E", "aliases": ["그린", "green"]},
    "기후": {"category": "E", "aliases": ["기후", "climate"]},
    "친환경": {"category": "E", "aliases": ["친환경", "eco-friendly"]},
    "환경": {"category": "E", "aliases": ["환경", "environment", "environmental"]},
    "탄소": {"category": "E", "aliases": ["탄소", "carbon"]},
    "넷제로": {"category": "E", "aliases": ["넷제로", "net-zero"]},
    "온실가스": {"category": "E", "aliases": ["온실가스", "greenhouse gas", "GHG"]},
    "에너지": {"category": "E", "aliases": ["에너지", "energy"]},
    "태양광": {"category": "E", "aliases": ["태양광"]},
    "풍력": {"category": "E", "aliases": ["풍력"]},
    "대기전력": {"category": "E", "aliases": ["대기전력"]},
    "자원": {"category": "E", "aliases": ["자원", "resource"]},
    "재활용": {"category": "E", "aliases": ["재활용", "recycling"]},
    "업사이클링": {"category": "E", "aliases": ["업사이클링", "upcycling"]},
    "분리배출": {"category": "E", "aliases": ["분리배출"]},
    "재사용": {"category": "E", "aliases": ["재사용", "reuse"]},
    "재판매": {"category": "E", "aliases": ["재판매", "resale"]},
    "제품수선": {"category": "E", "aliases": ["제품수선", "repair"]},
    "오염": {"category": "E", "aliases": ["오염", "pollution"]},
    "미세먼지": {"category": "E", "aliases": ["미세먼지"]},
    "플라스틱": {"category": "E", "aliases": ["플라스틱"]},
    "순환경제": {"category": "E", "aliases": ["순환경제", "circular economy"]},
    "동물복지": {"category": "E", "aliases": ["동물복지", "animal welfare"]},
    "폐기물": {"category": "E", "aliases": ["폐기물", "waste"]},
    "제로웨이스트": {"category": "E", "aliases": ["제로 웨이스트", "zero waste"]},
    "생물다양성": {"category": "E", "aliases": ["생물다양성", "biodiversity"]},
    "생태계": {"category": "E", "aliases": ["생태계", "ecosystems"]},
    "윤리적": {"category": "E", "aliases": ["윤리적", "ethical"]},
    "화학물질": {"category": "E", "aliases": ["화학물질"]},
    "유해물질": {"category": "E", "aliases": ["유해물질"]},
    "절약": {"category": "E", "aliases": ["절약"]},
    "감축": {"category": "E", "aliases": ["감축"]},
    "보전": {"category": "E", "aliases": ["보전"]},
    "감시": {"category": "E", "aliases": ["감시"]},

    # S
    "사회적책임": {"category": "S", "aliases": ["사회적 책임", "social responsibility"]},
    "책임": {"category": "S", "aliases": ["책임", "responsibility", "responsible"]},
    "노동": {"category": "S", "aliases": ["노동", "labor"]},
    "직원": {"category": "S", "aliases": ["직원", "employee"]},
    "권리": {"category": "S", "aliases": ["권리", "rights"]},
    "복지": {"category": "S", "aliases": ["복지", "welfare"]},
    "참여": {"category": "S", "aliases": ["참여", "engagement"]},
    "소비자": {"category": "S", "aliases": ["소비자", "consumer"]},
    "소비": {"category": "S", "aliases": ["소비", "consumption"]},
    "행동": {"category": "S", "aliases": ["행동", "behavior"]},
    "보호": {"category": "S", "aliases": ["보호", "protection"]},
    "신뢰": {"category": "S", "aliases": ["신뢰", "trust"]},
    "인권": {"category": "S", "aliases": ["인권", "human rights"]},
    "인적자본": {"category": "S", "aliases": ["인적 자본", "human capital"]},
    "다양성": {"category": "S", "aliases": ["다양성", "diversity"]},
    "평등": {"category": "S", "aliases": ["평등"]},
    "형평성": {"category": "S", "aliases": ["형평성", "equity"]},
    "포용성": {"category": "S", "aliases": ["포용성", "inclusion"]},
    "포용적": {"category": "S", "aliases": ["포용적", "inclusive"]},
    "지역사회": {"category": "S", "aliases": ["지역사회", "community"]},
    "개발": {"category": "S", "aliases": ["개발"]},
    "활동": {"category": "S", "aliases": ["활동"]},
    "의식": {"category": "S", "aliases": ["의식"]},
    "공헌": {"category": "S", "aliases": ["공헌"]},
    "건강": {"category": "S", "aliases": ["건강", "health"]},
    "안전": {"category": "S", "aliases": ["안전", "safety"]},
    "위생": {"category": "S", "aliases": ["위생", "sanitation"]},
    "고객": {"category": "S", "aliases": ["고객", "customer"]},
    "빈곤": {"category": "S", "aliases": ["빈곤", "poverty"]},
    "기아": {"category": "S", "aliases": ["기아", "hunger"]},
    "식량": {"category": "S", "aliases": ["식량", "food"]},
    "안보": {"category": "S", "aliases": ["안보", "security"]},
    "교육": {"category": "S", "aliases": ["교육", "education"]},
    "역량": {"category": "S", "aliases": ["역량", "upskilling"]},
    "훈련": {"category": "S", "aliases": ["훈련", "training"]},
    "성평등": {"category": "S", "aliases": ["성평등", "gender equity"]},
    "여성": {"category": "S", "aliases": ["여성", "women"]},
    "성격차": {"category": "S", "aliases": ["성 격차", "gender disparities"]},
    "불평등": {"category": "S", "aliases": ["불평등", "inequity"]},
    "공정": {"category": "S", "aliases": ["공정", "fair"]},
    "대우": {"category": "S", "aliases": ["대우", "treatment"]},
    "임금": {"category": "S", "aliases": ["임금", "wages"]},
    "기회": {"category": "S", "aliases": ["기회", "opportunity"]},
    "일자리": {"category": "S", "aliases": ["일자리", "work"]},
    "경제성장": {"category": "S", "aliases": ["경제 성장", "economic growth"]},
    "제품품질": {"category": "S", "aliases": ["제품 품질", "product quality"]},
    "제품수명주기": {"category": "S", "aliases": ["제품수명주기", "product life cycle"]},
    "공급망관리": {"category": "S", "aliases": ["공급망 관리", "supply chain management"]},
    "협력사관리": {"category": "S", "aliases": ["협력사 관리", "vendor management"]},
    "이해관계자": {"category": "S", "aliases": ["이해관계자", "stakeholders"]},
    "사회적가치": {"category": "S", "aliases": ["사회적 가치", "social value"]},
    "관계": {"category": "S", "aliases": ["관계"]},
    "돌봄": {"category": "S", "aliases": ["돌봄"]},
    "웰빙": {"category": "S", "aliases": ["웰빙", "well-being"]},
    "삶의질": {"category": "S", "aliases": ["삶의 질"]},
    "윤리": {"category": "S", "aliases": ["윤리", "ethic"]},
    "개인정보보호": {"category": "S", "aliases": ["개인정보 보호", "data privacy"]},
    "디지털시민성": {"category": "S", "aliases": ["디지털 시민성", "digital citizenship"]},

    # G
    "환경경영": {"category": "G", "aliases": ["ESG 환경경영", "환경경영"]},
    "ESG위원회": {"category": "G", "aliases": ["ESG 위원회"]},
    "기업가치": {"category": "G", "aliases": ["기업 가치", "firm value"]},
    "주주가치": {"category": "G", "aliases": ["주주 가치", "shareholder value"]},
    "투명성": {"category": "G", "aliases": ["투명성", "transparency"]},
    "공시": {"category": "G", "aliases": ["공시", "disclosure"]},
    "표시제도": {"category": "G", "aliases": ["표시제도"]},
    "반부패": {"category": "G", "aliases": ["반부패", "anti-corruption"]},
    "뇌물방지": {"category": "G", "aliases": ["뇌물방지"]},
    "법규준수": {"category": "G", "aliases": ["법규 준수", "legal compliance"]},
    "이사회": {"category": "G", "aliases": ["이사회", "board of directors"]},
    "독립성": {"category": "G", "aliases": ["독립성", "board independence"]},
    "감사위원회": {"category": "G", "aliases": ["감사위원회", "audit committee"]},
    "경영진보상": {"category": "G", "aliases": ["경영진 보상", "executive compensation"]},
    "주주권": {"category": "G", "aliases": ["주주권", "shareholder rights"]},
    "파트너십": {"category": "G", "aliases": ["파트너쉽", "partnership"]},
    "협력": {"category": "G", "aliases": ["협력", "collaboration"]},
    "기업지배구조": {"category": "G", "aliases": ["기업지배구조", "corporate governance"]},
    "가계관리": {"category": "G", "aliases": ["가계관리"]},
    "가계재무": {"category": "G", "aliases": ["가계재무"]},
    "소비계획": {"category": "G", "aliases": ["소비계획"]},
    "의사결정기준": {"category": "G", "aliases": ["의사결정 기준"]},
    "규칙": {"category": "G", "aliases": ["규칙"]},
    "정보활용": {"category": "G", "aliases": ["정보 활용"]},
    "정보판단": {"category": "G", "aliases": ["정보 판단"]},
}

# 복수 카테고리 정책은 단계2 집계 해석용으로 중요하지만,
# 단계3 연관분석에서는 canonical 기준 1차 대표 카테고리만 둔다.
# 다만 다중 해석이 필요한 토큰은 아래 메모용 dict에 유지한다.
KW_MULTI_CAT = {
    "복지": ["E", "S"],
    "보호": ["E", "S"],
    "행동": ["E", "S"],
    "윤리": ["S", "G"],
    "협력": ["G", "S"],
    "실천": ["E", "S"],
    "관리": ["E", "S"],
}

EXCLUDE_KWS = {"법", "물"}  # false positive 전면 제외 유지
ACRONYMS = {"ESG", "GRI", "TCFD", "SASB", "GHG", "ISSB"}

def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def normalize_no_space(text: str) -> str:
    return re.sub(r"\s+", "", str(text)).strip().lower()

def is_english_like(alias: str) -> bool:
    return bool(re.fullmatch(r"[A-Za-z0-9 .\-]+", alias))

def alias_to_pattern(alias: str):
    alias = normalize_space(alias)
    if alias in ACRONYMS:
        return ("regex", re.compile(rf"(?<![A-Za-z0-9]){re.escape(alias)}(?![A-Za-z0-9])"))
    if is_english_like(alias):
        escaped = re.escape(alias)
        escaped = escaped.replace(r"\ ", r"\s+")
        return ("regex", re.compile(rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])", flags=re.I))
    return ("nospace_substring", normalize_no_space(alias))

ESG_ALIAS_PATTERNS = {}
ALIAS_TO_CONCEPT = {}
CONCEPT_TO_CATEGORY = {}

for canonical, spec in ESG_CONCEPT_SPECS.items():
    CONCEPT_TO_CATEGORY[canonical] = spec["category"]
    for alias in spec["aliases"]:
        alias_norm = normalize_space(alias)
        ESG_ALIAS_PATTERNS[alias_norm] = alias_to_pattern(alias_norm)
        ALIAS_TO_CONCEPT[alias_norm.lower()] = canonical

ESG_RAW_ALIASES = list(ESG_ALIAS_PATTERNS.keys())
ESG_CANONICALS = list(ESG_CONCEPT_SPECS.keys())

print("raw alias 수:", len(ESG_RAW_ALIASES))
print("concept 수:", len(ESG_CANONICALS))


raw alias 수: 240
concept 수: 135


## Cell 4. 공통 함수

In [20]:

# =========================
# 3. 공통 함수
# =========================
def safe_literal_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
        # json list fallback
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
        return [t.strip() for t in re.split(r"[|,;/]+", s) if t.strip()]
    return []

def resolve_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"필수 컬럼 없음: {candidates}")
    return None

def split_sentences(text: str):
    text = str(text) if text is not None else ""
    text = text.strip()
    if not text:
        return []
    if HAS_KSS:
        try:
            sents = kss.split_sentences(text)
            sents = [s.strip() for s in sents if str(s).strip()]
            if sents:
                return sents
        except Exception:
            pass
    # fallback
    sents = re.split(r"(?<=[\.\!\?。！？])\s+|\n+", text)
    sents = [s.strip() for s in sents if s and s.strip()]
    return sents

#def match_aliases_in_sentence(sentence: str):
#    sent = normalize_space(sentence)
#    sent_ns = normalize_no_space(sentence)
#    matched_aliases = set()
#    for alias, (mode, obj) in ESG_ALIAS_PATTERNS.items():
#        if mode == "regex":
#            if obj.search(sent):
#                matched_aliases.add(alias)
#        else:
#            if obj and obj in sent_ns:
#                matched_aliases.add(alias)
#    # false positive 전면 제외
#    matched_aliases = {a for a in matched_aliases if normalize_space(a) not in EXCLUDE_KWS}
#    return matched_aliases


# =========================
# [Stage III 수정]
# 중첩 alias 제거용 helper
# =========================
import re

def find_all_nonregex_candidates(sentence):
    """
    공백 제거(compact) 문자열 기준으로
    non-regex alias의 모든 매칭 후보(span)를 수집
    """
    sent_ns = normalize_no_space(sentence)
    candidates = []

    for alias, (mode, obj) in ESG_ALIAS_PATTERNS.items():
        if mode != "regex":
            alias_ns = obj  # alias_to_pattern()에서 non-regex면 compact 문자열이 들어있음
            if not alias_ns:
                continue

            for m in re.finditer(re.escape(alias_ns), sent_ns):
                candidates.append({
                    "alias": alias,
                    "mode": "compact",
                    "start": m.start(),
                    "end": m.end(),
                    "length": m.end() - m.start(),
                })

    return candidates


def find_all_regex_candidates(sentence):
    """
    regex alias는 기존 방식 유지
    """
    sent = normalize_space(sentence)
    candidates = []

    for alias, (mode, obj) in ESG_ALIAS_PATTERNS.items():
        if mode == "regex":
            for m in obj.finditer(sent):
                candidates.append({
                    "alias": alias,
                    "mode": "regex",
                    # compact span과 충돌 비교 안 하도록 별도 좌표계 사용
                    "start": 10**9 + m.start(),
                    "end": 10**9 + m.end(),
                    "length": m.end() - m.start(),
                })

    return candidates


def greedy_select_non_overlapping(candidates):
    """
    긴 표현 우선, 같은 좌표계 내 overlapping 후보 제거
    """
    if not candidates:
        return []

    # 긴 표현 우선 -> 시작점 -> alias
    candidates = sorted(
        candidates,
        key=lambda x: (-x["length"], x["start"], x["alias"])
    )

    accepted = []
    occupied = []

    for c in candidates:
        overlaps = False
        for s, e, mode in occupied:
            if c["mode"] != mode:
                continue
            if not (c["end"] <= s or c["start"] >= e):
                overlaps = True
                break

        if not overlaps:
            accepted.append(c)
            occupied.append((c["start"], c["end"], c["mode"]))

    # 문장 내 최종 alias 집합은 중복 제거 후 반환
    return sorted({x["alias"] for x in accepted})


def match_aliases_in_sentence(sentence: str):
    """
    수정 버전:
    - regex alias: 기존 유지
    - compact alias: longest-match + non-overlap
    - 동일 표면형 내부의 중첩 alias 제거
    """
    regex_cands = find_all_regex_candidates(sentence)
    compact_cands = find_all_nonregex_candidates(sentence)

    all_cands = regex_cands + compact_cands
    matched_aliases = greedy_select_non_overlapping(all_cands)

    # false positive 전면 제외
    matched_aliases = {
        a for a in matched_aliases
        if normalize_space(a) not in EXCLUDE_KWS
    }

    return matched_aliases


OPEN_VOCAB_STOPWORDS = {
    "대학교", "대학원", "학과", "전공", "교육", "교과목", "연구", "학생", "교수", "관련",
    "소개", "과정", "운영", "기반", "통해", "위한", "경우", "대한", "및", "에서", "으로",
    "있는", "있습니다", "합니다", "등의", "수업", "학부", "대학", "전문", "프로그램", "센터",
    "공지", "게시판", "자료", "문의", "실습", "현장", "국내", "국제", "분야", "이론",
    "실무", "융합", "최신", "중심", "강의", "학문", "역량", "강화", "교육과정", "개설",
    "수행", "지도", "대상", "지원", "관리", "활동", "개발", "관계", "행동", "복지", "보호",
}

def extract_open_vocab_tokens(sentence: str, doc_nouns=None):
    sent_ns = normalize_no_space(sentence)
    out = []

    # 1) 우선 단계1 nouns_filtered를 재사용
    if doc_nouns:
        for tok in doc_nouns:
            tok = str(tok).strip()
            if not tok:
                continue
            t = tok.lower()
            if len(t) < 2:
                continue
            if t in {a.lower() for a in ESG_RAW_ALIASES}:
                continue
            if t in {c.lower() for c in ESG_CANONICALS}:
                continue
            if t in {w.lower() for w in OPEN_VOCAB_STOPWORDS}:
                continue
            if normalize_no_space(tok) in sent_ns:
                out.append(t)

    # 2) nouns_filtered가 빈 경우만 regex fallback
    if not out:
        tokens = re.findall(r"[가-힣A-Za-z][가-힣A-Za-z0-9_+\-]{1,}", str(sentence))
        for tok in tokens:
            t = tok.strip().lower()
            if not t or t.isdigit():
                continue
            if len(t) < 2:
                continue
            if t in {a.lower() for a in ESG_RAW_ALIASES}:
                continue
            if t in {c.lower() for c in ESG_CANONICALS}:
                continue
            if t in {w.lower() for w in OPEN_VOCAB_STOPWORDS}:
                continue
            # 조사/어미가 붙은 한국어 fallback 제거
            if re.search(r"[가-힣](은|는|이|가|을|를|와|과|도|에|의|로|으로|에서|하다|한다|됨|된)$", t):
                continue
            out.append(t)

    return sorted(set(out))

def ochiai(n_xy, n_x, n_y):
    denom = math.sqrt(max(n_x, 0) * max(n_y, 0))
    if denom == 0:
        return 0.0
    return n_xy / denom

def top_sentence_samples(df_sentence, anchor_col, related_col, anchor, related, n=5):
    mask = (
        df_sentence[anchor_col].apply(lambda x: anchor in x) &
        df_sentence[related_col].apply(lambda x: related in x)
    )
    cols = ["doc_id", "sentence_id", "university", "department", "page_type", "sentence"]
    out = df_sentence.loc[mask, cols].drop_duplicates().head(n).copy()
    return out

def ensure_serializable_list(x):
    if isinstance(x, set):
        return sorted(x)
    if isinstance(x, list):
        return x
    return list(x)

def save_multi_csv(sample_dict, out_dir: Path, prefix: str):
    out_dir.mkdir(parents=True, exist_ok=True)
    meta = []
    for name, df_ in sample_dict.items():
        fp = out_dir / f"{prefix}_{name}.csv"
        df_.to_csv(fp, index=False, encoding="utf-8-sig")
        meta.append({"name": name, "path": str(fp), "n_rows": len(df_)})
    pd.DataFrame(meta).to_csv(out_dir / f"{prefix}_index.csv", index=False, encoding="utf-8-sig")


## Cell 5. 데이터 로드

In [21]:

# =========================
# 4. 데이터 로드
# =========================
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    return df

def prepare_df(df):
    text_col = resolve_col(df, ["text", "content", "page_text"])
    univ_col = resolve_col(df, ["university", "school", "대학명"], required=False)
    dept_col = resolve_col(df, ["department", "major", "학과"], required=False)
    ptype_col = resolve_col(df, ["page_type", "pageType"], required=False)
    dual_col = resolve_col(df, ["crawl_dual", "page_category", "page_group"], required=False)
    url_col = resolve_col(df, ["url", "link"], required=False)
    title_col = resolve_col(df, ["title", "page_title"], required=False)
    nouns_col = resolve_col(df, ["nouns_filtered", "nouns_list"], required=False)
    bigrams_col = resolve_col(df, ["sent_bigrams_list"], required=False)

    out = df.copy()
    out["doc_id"] = np.arange(1, len(out) + 1)
    out["text"] = out[text_col].fillna("").astype(str)
    out["university"] = out[univ_col].fillna("NA").astype(str) if univ_col else "NA"
    out["department"] = out[dept_col].fillna("NA").astype(str) if dept_col else "NA"
    out["page_type"] = out[ptype_col].fillna("NA").astype(str) if ptype_col else "NA"
    out["crawl_dual"] = out[dual_col].fillna("NA").astype(str) if dual_col else "NA"
    out["url"] = out[url_col].fillna("").astype(str) if url_col else ""
    out["title"] = out[title_col].fillna("").astype(str) if title_col else ""
    out["nouns_filtered"] = out[nouns_col].apply(safe_literal_list) if nouns_col else [[] for _ in range(len(out))]
    out["sent_bigrams_list"] = out[bigrams_col].apply(safe_literal_list) if bigrams_col else [[] for _ in range(len(out))]
    out["char_len"] = out["text"].str.len()

    return out[[
        "doc_id", "university", "department", "page_type", "crawl_dual",
        "url", "title", "text", "char_len", "nouns_filtered", "sent_bigrams_list"
    ]]

loaded = {}
for scope, path in RUN_CONFIG.items():
    df_raw = load_jsonl(path)
    loaded[scope] = prepare_df(df_raw)

for scope, df in loaded.items():
    print(scope, df.shape, "문서수:", len(df), "비어있지 않은 text:", (df["text"].str.strip() != "").sum())


all (1054, 11) 문서수: 1054 비어있지 않은 text: 1054
main (866, 11) 문서수: 866 비어있지 않은 text: 866


## Cell 6. 문장 단위 테이블 생성

In [22]:
# =========================
# 5. 문장 단위 테이블 생성 (최종 실행본)
# =========================
from kss import split_sentences
from tqdm.auto import tqdm
import pandas as pd
import gc
import time

def build_sentence_level_df_chunked(
    df,
    backend="fast",
    num_workers=1,
    chunk_size=5,
    verbose=True
):
    required_cols = [
        "doc_id", "university", "department", "page_type",
        "crawl_dual", "url", "title", "nouns_filtered", "text"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}")

    metas = df[required_cols].to_dict("records")
    total_docs = len(metas)

    chunk_frames = []
    total_sentence_count = 0
    total_esg_sentence_count = 0

    for chunk_idx, start in enumerate(
        tqdm(range(0, total_docs, chunk_size), desc="문서 chunk 처리"),
        start=1
    ):
        end = min(start + chunk_size, total_docs)
        remaining_docs = total_docs - end
        chunk = metas[start:end]

        texts = [
            m["text"] if isinstance(m["text"], str) else ""
            for m in chunk
        ]

        if verbose:
            print(
                f"\n[chunk {chunk_idx}] split 시작 | "
                f"문서 {start+1:,}~{end:,}/{total_docs:,} | "
                f"남은 문서 {remaining_docs:,}개"
            )

        split_results = split_sentences(
            texts,
            backend=backend,
            num_workers=num_workers
        )

        if verbose:
            print(f"[chunk {chunk_idx}] split 완료")

        chunk_records = []
        chunk_sentence_count = 0
        chunk_esg_sentence_count = 0

        for meta, sents in zip(chunk, split_results):
            if not sents:
                continue

            for sid, sent in enumerate(sents, start=1):
                raw_aliases = sorted(match_aliases_in_sentence(sent))
                concept_matches = sorted({
                    ALIAS_TO_CONCEPT[a.lower()]
                    for a in raw_aliases
                })
                categories = sorted({
                    CONCEPT_TO_CATEGORY[c]
                    for c in concept_matches
                    if c in CONCEPT_TO_CATEGORY
                })
                open_vocab = extract_open_vocab_tokens(sent, meta["nouns_filtered"])

                n_concept = len(concept_matches)

                chunk_records.append({
                    "doc_id": meta["doc_id"],
                    "sentence_id": sid,
                    "university": meta["university"],
                    "department": meta["department"],
                    "page_type": meta["page_type"],
                    "crawl_dual": meta["crawl_dual"],
                    "url": meta["url"],
                    "title": meta["title"],
                    "sentence": sent,
                    "raw_aliases": raw_aliases,
                    "concept_matches": concept_matches,
                    "categories": categories,
                    "open_vocab": open_vocab,
                    "n_raw": len(raw_aliases),
                    "n_concept": n_concept,
                })

                chunk_sentence_count += 1
                if n_concept > 0:
                    chunk_esg_sentence_count += 1

        chunk_df = pd.DataFrame(chunk_records)
        chunk_frames.append(chunk_df)

        total_sentence_count += chunk_sentence_count
        total_esg_sentence_count += chunk_esg_sentence_count

        if verbose:
            print(
                f"[chunk {chunk_idx}] 완료 | "
                f"누적 처리 문서 {end:,}/{total_docs:,} | "
                f"남은 문서 {remaining_docs:,}개 | "
                f"누적 문장 {total_sentence_count:,}개 | "
                f"누적 ESG 매칭 문장 {total_esg_sentence_count:,}개"
            )

        # 메모리 정리
        del chunk_records, chunk_df, split_results, texts, chunk
        gc.collect()

    sent_df = pd.concat(chunk_frames, ignore_index=True) if chunk_frames else pd.DataFrame()

    if verbose and not sent_df.empty:
        print(
            f"\n최종 완료 | 전체 문서 {total_docs:,}개 | "
            f"전체 문장 {len(sent_df):,}개 | "
            f"ESG 매칭 문장 {(sent_df['n_concept'] > 0).sum():,}개"
        )

    del chunk_frames, metas
    gc.collect()

    return sent_df


# =========================
# 5-1. 실행 함수
# =========================
def run_sentence_table(scope_name, df, backend="fast", num_workers=1, chunk_size=5, verbose=True):
    print(f"\n===== {scope_name} sentence table 생성 시작 =====")
    t0 = time.time()

    sent_df = build_sentence_level_df_chunked(
        df=df,
        backend=backend,
        num_workers=num_workers,
        chunk_size=chunk_size,
        verbose=verbose
    )

    elapsed = time.time() - t0
    print(
        f"===== {scope_name} 완료 | shape={sent_df.shape} | "
        f"ESG 매칭 문장={(sent_df['n_concept'] > 0).sum() if not sent_df.empty else 0:,} | "
        f"소요시간={elapsed/60:.2f}분 ====="
    )
    return sent_df

In [23]:

# =========================
# 5-2. 실제 실행
# main 먼저, 이후 all 실행
# =========================
sentence_tables = {}

# 1) main
sentence_tables["main"] = run_sentence_table(
    scope_name="main",
    df=loaded["main"],
    backend="fast",
    num_workers=1,
    chunk_size=100,
    verbose=True
)

# 2) all
sentence_tables["all"] = run_sentence_table(
    scope_name="all",
    df=loaded["all"],
    backend="fast",
    num_workers=1,
    chunk_size=100,
    verbose=True
)

# 확인
for scope, sent_df in sentence_tables.items():
    print(
        scope,
        sent_df.shape,
        "ESG 매칭 문장:",
        (sent_df["n_concept"] > 0).sum() if not sent_df.empty else 0
    )


===== main sentence table 생성 시작 =====


문서 chunk 처리:   0%|          | 0/9 [00:00<?, ?it/s]


[chunk 1] split 시작 | 문서 1~100/866 | 남은 문서 766개
[chunk 1] split 완료
[chunk 1] 완료 | 누적 처리 문서 100/866 | 남은 문서 766개 | 누적 문장 1,119개 | 누적 ESG 매칭 문장 815개


문서 chunk 처리:  11%|█         | 1/9 [00:52<07:00, 52.56s/it]


[chunk 2] split 시작 | 문서 101~200/866 | 남은 문서 666개
[chunk 2] split 완료


...


[chunk 10] split 시작 | 문서 901~1,000/1,054 | 남은 문서 54개
[chunk 10] split 완료
[chunk 10] 완료 | 누적 처리 문서 1,000/1,054 | 남은 문서 54개 | 누적 문장 7,941개 | 누적 ESG 매칭 문장 4,516개


문서 chunk 처리:  91%|█████████ | 10/11 [05:14<00:26, 26.08s/it]


[chunk 11] split 시작 | 문서 1,001~1,054/1,054 | 남은 문서 0개
[chunk 11] split 완료
[chunk 11] 완료 | 누적 처리 문서 1,054/1,054 | 남은 문서 0개 | 누적 문장 8,412개 | 누적 ESG 매칭 문장 4,825개


문서 chunk 처리: 100%|██████████| 11/11 [05:22<00:00, 29.31s/it]



최종 완료 | 전체 문서 1,054개 | 전체 문장 8,412개 | ESG 매칭 문장 4,825개
===== all 완료 | shape=(8412, 15) | ESG 매칭 문장=4,825 | 소요시간=5.38분 =====
main (7243, 15) ESG 매칭 문장: 4232
all (8412, 15) ESG 매칭 문장: 4825


## Cell 7. ESG-only 연관분석

In [24]:
# =========================
# 6. ESG-only 연관분석
# =========================
def compute_esg_only_edges(df_sentence, key_col):
    # key_col: raw_aliases or concept_matches
    anchor_sent_n = Counter()
    pair_sent_n = Counter()
    pair_univ = defaultdict(set)
    pair_ptype = defaultdict(set)
    pair_cat_left = {}
    pair_cat_right = {}

    for row in df_sentence.itertuples(index=False):
        items = sorted(set(getattr(row, key_col)))
        if len(items) == 0:
            continue
        for x in items:
            if key_col == "raw_aliases":
                c = ALIAS_TO_CONCEPT[x.lower()]
                cat = CONCEPT_TO_CATEGORY[c]
            else:
                cat = CONCEPT_TO_CATEGORY[x]
            anchor_sent_n[x] += 1
            pair_cat_left[x] = cat
            pair_cat_right[x] = cat

        for a, b in combinations(items, 2):
            pair = tuple(sorted((a, b)))
            pair_sent_n[pair] += 1
            pair_univ[pair].add(row.university)
            pair_ptype[pair].add(row.page_type)

    rows = []
    for (a, b), n_xy in pair_sent_n.items():
        n_x = anchor_sent_n[a]
        n_y = anchor_sent_n[b]
        univ_n = len(pair_univ[(a, b)])
        if n_x < MIN_ANCHOR_SENT_N or n_y < MIN_ANCHOR_SENT_N:
            continue
        if n_xy < MIN_CO_SENT_N:
            continue
        if univ_n < MIN_UNIV_N:
            continue
        rows.append({
            "anchor": a,
            "related": b,
            "category_anchor": pair_cat_left[a],
            "category_related": pair_cat_right[b],
            "anchor_sent_n": n_x,
            "related_sent_n": n_y,
            "co_sent_n": n_xy,
            "ochiai": ochiai(n_xy, n_x, n_y),
            "university_n": univ_n,
            "page_type_n": len(pair_ptype[(a, b)]),
            "universities": sorted(pair_univ[(a, b)]),
            "page_types": sorted(pair_ptype[(a, b)]),
        })
    edge_df = pd.DataFrame(rows).sort_values(
        ["ochiai", "co_sent_n", "anchor", "related"],
        ascending=[False, False, True, True]
    ).reset_index(drop=True)

    node_rows = []
    for node, n in anchor_sent_n.items():
        if n >= MIN_ANCHOR_SENT_N:
            if key_col == "raw_aliases":
                canonical = ALIAS_TO_CONCEPT[node.lower()]
                cat = CONCEPT_TO_CATEGORY[canonical]
            else:
                canonical = node
                cat = CONCEPT_TO_CATEGORY[node]
            node_rows.append({
                "node": node,
                "canonical": canonical,
                "category": cat,
                "sent_n": n,
            })
    node_df = pd.DataFrame(node_rows).sort_values(["sent_n", "node"], ascending=[False, True]).reset_index(drop=True)
    return edge_df, node_df

esg_only_results = {}
for scope, sent_df in sentence_tables.items():
    raw_edges, raw_nodes = compute_esg_only_edges(sent_df, "raw_aliases")
    concept_edges, concept_nodes = compute_esg_only_edges(sent_df, "concept_matches")
    esg_only_results[scope] = {
        "raw_edges": raw_edges,
        "raw_nodes": raw_nodes,
        "concept_edges": concept_edges,
        "concept_nodes": concept_nodes,
    }
    print(scope, "raw_edges:", len(raw_edges), "concept_edges:", len(concept_edges))


main raw_edges: 1954 concept_edges: 1177
all raw_edges: 1994 concept_edges: 1214


## Cell 8. open-vocab 연관분석

In [25]:

# =========================
# 7. open-vocab 연관분석
# =========================
def compute_open_vocab_edges(df_sentence, anchor_col="concept_matches", related_col="open_vocab"):
    anchor_sent_n = Counter()
    related_sent_n = Counter()
    pair_sent_n = Counter()
    pair_univ = defaultdict(set)
    pair_ptype = defaultdict(set)

    for row in df_sentence.itertuples(index=False):
        anchors = sorted(set(getattr(row, anchor_col)))
        relateds = sorted(set(getattr(row, related_col)))
        if not anchors or not relateds:
            continue

        for a in anchors:
            anchor_sent_n[a] += 1
        for r in relateds:
            related_sent_n[r] += 1

        for a in anchors:
            for r in relateds:
                pair = (a, r)
                pair_sent_n[pair] += 1
                pair_univ[pair].add(row.university)
                pair_ptype[pair].add(row.page_type)

    rows = []
    for (a, r), n_xy in pair_sent_n.items():
        n_a = anchor_sent_n[a]
        n_r = related_sent_n[r]
        univ_n = len(pair_univ[(a, r)])
        if n_a < MIN_ANCHOR_SENT_N:
            continue
        if n_xy < MIN_CO_SENT_N:
            continue
        if univ_n < MIN_UNIV_N:
            continue
        rows.append({
            "anchor": a,
            "related": r,
            "category_anchor": CONCEPT_TO_CATEGORY.get(a, "NA"),
            "anchor_sent_n": n_a,
            "related_sent_n": n_r,
            "co_sent_n": n_xy,
            "ochiai": ochiai(n_xy, n_a, n_r),
            "university_n": univ_n,
            "page_type_n": len(pair_ptype[(a, r)]),
            "universities": sorted(pair_univ[(a, r)]),
            "page_types": sorted(pair_ptype[(a, r)]),
        })
    edge_df = pd.DataFrame(rows).sort_values(
        ["anchor", "ochiai", "co_sent_n", "related"],
        ascending=[True, False, False, True]
    ).reset_index(drop=True)

    if len(edge_df) == 0:
        return edge_df, edge_df.copy()

    topk = (
        edge_df.groupby("anchor", group_keys=False)
        .head(TOPN_OPEN_VOCAB)
        .reset_index(drop=True)
    )

    fp_cands = edge_df[
        edge_df["related"].isin(["교육", "연구", "학생", "교수", "과정", "학과", "대학"])
    ].copy()

    return topk, fp_cands

open_vocab_results = {}
for scope, sent_df in sentence_tables.items():
    topk, fp_cands = compute_open_vocab_edges(sent_df)
    open_vocab_results[scope] = {
        "topk": topk,
        "fp_candidates": fp_cands,
    }
    print(scope, "open_vocab_rows:", len(topk))


main open_vocab_rows: 3035
all open_vocab_rows: 3098


## Cell 9. 요약표

In [26]:

# =========================
# 8. 요약표
# =========================
def build_summary_tables(df_doc, df_sent, esg_res, ov_res):
    doc_summary = pd.DataFrame([{
        "doc_n": len(df_doc),
        "nonempty_text_n": (df_doc["text"].str.strip() != "").sum(),
        "sentence_n": len(df_sent),
        "esg_sentence_n_raw": (df_sent["n_raw"] > 0).sum(),
        "esg_sentence_n_concept": (df_sent["n_concept"] > 0).sum(),
        "university_n": df_doc["university"].nunique(),
        "department_n": df_doc["department"].nunique(),
        "page_type_n": df_doc["page_type"].nunique(),
        "mean_char_len": round(df_doc["char_len"].mean(), 2),
        "median_char_len": round(df_doc["char_len"].median(), 2),
    }])

    by_univ = (
        df_sent.assign(has_esg=df_sent["n_concept"] > 0)
        .groupby("university", dropna=False)
        .agg(
            sentence_n=("sentence", "size"),
            esg_sentence_n=("has_esg", "sum"),
            page_type_n=("page_type", "nunique"),
        )
        .reset_index()
    )
    by_univ["esg_sentence_ratio"] = by_univ["esg_sentence_n"] / by_univ["sentence_n"]

    by_ptype = (
        df_sent.assign(has_esg=df_sent["n_concept"] > 0)
        .groupby("page_type", dropna=False)
        .agg(
            sentence_n=("sentence", "size"),
            esg_sentence_n=("has_esg", "sum"),
            university_n=("university", "nunique"),
        )
        .reset_index()
    )
    by_ptype["esg_sentence_ratio"] = by_ptype["esg_sentence_n"] / by_ptype["sentence_n"]

    raw_top = esg_res["raw_edges"].head(50).copy()
    concept_top = esg_res["concept_edges"].head(50).copy()
    open_top = ov_res["topk"].copy()

    return {
        "summary_doc": doc_summary,
        "summary_by_university": by_univ.sort_values(["esg_sentence_ratio", "esg_sentence_n"], ascending=[False, False]),
        "summary_by_page_type": by_ptype.sort_values(["esg_sentence_ratio", "esg_sentence_n"], ascending=[False, False]),
        "esg_only_raw_top50": raw_top,
        "esg_only_concept_top50": concept_top,
        "open_vocab_top": open_top,
    }

summary_tables = {}
for scope in loaded:
    summary_tables[scope] = build_summary_tables(
        loaded[scope], sentence_tables[scope], esg_only_results[scope], open_vocab_results[scope]
    )
    print(scope, "summary 준비 완료")


all summary 준비 완료
main summary 준비 완료


## Cell 10. QA용 예시 문장 추출

In [27]:

# =========================
# 9. QA용 예시 문장 추출
# =========================
qa_samples = {}
for scope, sent_df in sentence_tables.items():
    samples = {}

    raw_edges = esg_only_results[scope]["raw_edges"].head(15)
    for i, row in raw_edges.iterrows():
        key = f"raw_edge_{i+1:02d}_{row['anchor']}__{row['related']}"
        samples[key] = top_sentence_samples(
            sent_df, "raw_aliases", "raw_aliases", row["anchor"], row["related"], TOPN_SENTENCE_SAMPLE
        )

    concept_edges = esg_only_results[scope]["concept_edges"].head(15)
    for i, row in concept_edges.iterrows():
        key = f"concept_edge_{i+1:02d}_{row['anchor']}__{row['related']}"
        samples[key] = top_sentence_samples(
            sent_df, "concept_matches", "concept_matches", row["anchor"], row["related"], TOPN_SENTENCE_SAMPLE
        )

    open_edges = open_vocab_results[scope]["topk"].head(15)
    for i, row in open_edges.iterrows():
        key = f"open_vocab_{i+1:02d}_{row['anchor']}__{row['related']}"
        samples[key] = top_sentence_samples(
            sent_df, "concept_matches", "open_vocab", row["anchor"], row["related"], TOPN_SENTENCE_SAMPLE
        )

    qa_samples[scope] = samples
    print(scope, "qa sample 테이블 수:", len(samples))


main qa sample 테이블 수: 45
all qa sample 테이블 수: 45


## Cell 11. 저장

In [28]:

# =========================
# 10. 저장
# =========================
def save_scope_outputs(scope, df_doc, df_sent, out_dir: Path):
    xlsx_path = out_dir / f"연관분석결과_{scope}.xlsx"
    sample_dir = out_dir / f"qa_samples_{scope}"
    sample_dir.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        # README
        readme = pd.DataFrame({
            "항목": [
                "scope", "입력문서수", "문장수", "공출현단위", "지표", "raw/concept 분리", 
                "MIN_ANCHOR_SENT_N", "MIN_CO_SENT_N", "MIN_UNIV_N",
                "해석 주의"
            ],
            "내용": [
                scope, len(df_doc), len(df_sent), "문장 단위", "Ochiai",
                "예", MIN_ANCHOR_SENT_N, MIN_CO_SENT_N, MIN_UNIV_N,
                "연관성은 공출현 경향이지 인과나 실천수준이 아님"
            ]
        })
        readme.to_excel(writer, sheet_name="README", index=False)

        for sheet, sdf in summary_tables[scope].items():
            sdf.to_excel(writer, sheet_name=sheet[:31], index=False)

        esg_only_results[scope]["raw_nodes"].to_excel(writer, sheet_name="esg_raw_nodes", index=False)
        esg_only_results[scope]["raw_edges"].to_excel(writer, sheet_name="esg_raw_edges", index=False)
        esg_only_results[scope]["concept_nodes"].to_excel(writer, sheet_name="esg_concept_nodes", index=False)
        esg_only_results[scope]["concept_edges"].to_excel(writer, sheet_name="esg_concept_edges", index=False)

        open_vocab_results[scope]["topk"].to_excel(writer, sheet_name="open_vocab_topk", index=False)
        open_vocab_results[scope]["fp_candidates"].to_excel(writer, sheet_name="open_vocab_fp", index=False)

        # sentence QA overview
        qa_index = pd.DataFrame({
            "sample_name": list(qa_samples[scope].keys()),
            "n_rows": [len(v) for v in qa_samples[scope].values()]
        })
        qa_index.to_excel(writer, sheet_name="qa_index", index=False)

    save_multi_csv(qa_samples[scope], sample_dir, f"qa_{scope}")

    # sentence-level lightweight save
    sent_export = df_sent[[
        "doc_id", "sentence_id", "university", "department", "page_type", "crawl_dual",
        "sentence", "raw_aliases", "concept_matches", "categories", "open_vocab"
    ]].copy()
    sent_export["raw_aliases"] = sent_export["raw_aliases"].apply(lambda x: "|".join(x))
    sent_export["concept_matches"] = sent_export["concept_matches"].apply(lambda x: "|".join(x))
    sent_export["categories"] = sent_export["categories"].apply(lambda x: "|".join(x))
    sent_export["open_vocab"] = sent_export["open_vocab"].apply(lambda x: "|".join(x))
    sent_export.to_csv(out_dir / f"sentence_level_{scope}.csv", index=False, encoding="utf-8-sig")

    return xlsx_path

saved_files = []
for scope in loaded:
    fp = save_scope_outputs(scope, loaded[scope], sentence_tables[scope], OUTPUT_DIR)
    saved_files.append(fp)

saved_files


[WindowsPath('C:/Users/legen/Desktop/Lab Project/ESG/v2/단계3_연관분석/연관분석결과_all.xlsx'),
 WindowsPath('C:/Users/legen/Desktop/Lab Project/ESG/v2/단계3_연관분석/연관분석결과_main.xlsx')]

## Cell 12. 점검

In [29]:

# =========================
# 11. 실행 후 점검
# =========================
for scope in loaded:
    print("=" * 80)
    print(f"[{scope}]")
    print("문서수:", len(loaded[scope]))
    print("문장수:", len(sentence_tables[scope]))
    print("raw edge 수:", len(esg_only_results[scope]["raw_edges"]))
    print("concept edge 수:", len(esg_only_results[scope]["concept_edges"]))
    print("open-vocab row 수:", len(open_vocab_results[scope]["topk"]))
    print()
    print("[concept top 10]")
    display(esg_only_results[scope]["concept_edges"].head(10))


[all]
문서수: 1054
문장수: 8412
raw edge 수: 1994
concept edge 수: 1214
open-vocab row 수: 3098

[concept top 10]


,anchor,related,category_anchor,category_related,anchor_sent_n,related_sent_n,co_sent_n,ochiai,university_n,page_type_n,universities,page_types
0,소비,소비자,S,S,313,865,180,0.345933,13,8,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅂ, ㅈ, ㅊ, ㅋ, ㅌ]"
1,안전,위생,S,S,298,163,74,0.335761,13,9,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 동덕여자대학교, 서울대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅁ, ㅂ, ㅇ, ㅈ, ㅊ, ㅋ]"
2,성평등,인권,S,S,14,17,5,0.324102,2,3,"[고려대학교, 한국교원대학교]","[ㄷ, ㄹ, ㅋ]"
3,건강,지역사회,S,S,551,166,89,0.294280,15,7,"[경희대학교, 고려대학교, 단국대학교, 덕성여자대학교, 동덕여자대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅂ, ㅈ, ㅊ, ㅋ]"
4,신뢰,행동,S,S,46,93,19,0.290491,6,1,"[서울대학교, 성균관대학교, 연세대학교, 이화여자대학교, 인하대학교, 한양대학교]",[ㄹ]
5,지속가능,지속가능성,E,E,197,70,34,0.289532,11,6,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 서울대학교, 성균관대학교, 연세대학교, 이화여자대학교, 인하대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅁ, ㅂ, ㅋ]"
6,재활용,플라스틱,E,E,12,10,3,0.273861,3,3,"[단국대학교, 성균관대학교, 이화여자대학교]","[ㄹ, ㅂ, ㅇ]"
7,개발,교육,S,S,881,1865,338,0.263687,18,10,"[건국대학교, 경인교육대학교, 경희대학교, 고려대학교, 단국대학교, 덕성여자대학교, 동덕여자대학교, 서울교육대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅁ, ㅂ, ㅅ, ㅈ, ㅊ, ㅋ, ㅌ]"
8,개발,건강,S,S,881,551,182,0.261221,17,10,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 덕성여자대학교, 동덕여자대학교, 서울교육대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅂ, ㅅ, ㅇ, ㅈ, ㅊ, ㅋ, ㅌ]"
9,ESG,신뢰,ESG,S,16,46,7,0.258023,3,2,"[서울대학교, 성균관대학교, 이화여자대학교]","[ㄹ, ㅂ]"


[main]
문서수: 866
문장수: 7243
raw edge 수: 1954
concept edge 수: 1177
open-vocab row 수: 3035

[concept top 10]


,anchor,related,category_anchor,category_related,anchor_sent_n,related_sent_n,co_sent_n,ochiai,university_n,page_type_n,universities,page_types
0,소비,소비자,S,S,305,793,176,0.357871,13,5,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅂ, ㅈ]"
1,안전,위생,S,S,272,155,70,0.340916,13,7,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 동덕여자대학교, 서울대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅁ, ㅂ, ㅇ, ㅈ]"
2,신뢰,행동,S,S,44,91,19,0.300266,6,1,"[서울대학교, 성균관대학교, 연세대학교, 이화여자대학교, 인하대학교, 한양대학교]",[ㄹ]
3,건강,지역사회,S,S,515,159,85,0.297041,15,5,"[경희대학교, 고려대학교, 단국대학교, 덕성여자대학교, 동덕여자대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅂ, ㅈ]"
4,지속가능,지속가능성,E,E,179,69,33,0.296936,11,5,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 서울대학교, 성균관대학교, 연세대학교, 이화여자대학교, 인하대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅁ, ㅂ]"
5,ESG,신뢰,ESG,S,13,44,7,0.292685,3,2,"[서울대학교, 성균관대학교, 이화여자대학교]","[ㄹ, ㅂ]"
6,성평등,인권,S,S,10,11,3,0.286039,2,2,"[고려대학교, 한국교원대학교]","[ㄷ, ㄹ]"
7,재활용,플라스틱,E,E,12,10,3,0.273861,3,3,"[단국대학교, 성균관대학교, 이화여자대학교]","[ㄹ, ㅂ, ㅇ]"
8,개발,교육,S,S,833,1559,307,0.269397,18,7,"[건국대학교, 경인교육대학교, 경희대학교, 고려대학교, 단국대학교, 덕성여자대학교, 동덕여자대학교, 서울교육대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅁ, ㅂ, ㅅ, ㅈ]"
9,개발,건강,S,S,833,515,174,0.265658,16,7,"[건국대학교, 경희대학교, 고려대학교, 단국대학교, 덕성여자대학교, 동덕여자대학교, 서울대학교, 서울여자대학교, 성균관대학교, 숙명여자대학교, 연세대학교, 이화여자대학교, 인하대학교, 중앙대학교, 한국교원대학교, 한양대학교]","[ㄱ, ㄷ, ㄹ, ㅂ, ㅅ, ㅇ, ㅈ]"


## Cell 13. QA용 예시 문장 추출 (최종 수정안)

In [30]:
# =========================
# 13. QA용 예시 문장 추출 (최종 수정안)
# - hit sentence
# - 앞뒤 1문장(local context)
# - title / page_type 포함
# =========================
import os
import re
import gc
import pandas as pd

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def normalize_filename(text):
    text = str(text)
    text = re.sub(r'[\\/:*?"<>|]+', "_", text)
    text = re.sub(r"\s+", "_", text).strip("_")
    return text[:120]

def build_sentence_context_table(sent_df, window=1):
    """
    sentence_tables[scope]에서
    각 hit sentence에 대해 앞/뒤 문장과 local context를 붙일 수 있는 lookup table 생성
    """
    required_cols = ["doc_id", "sentence_id", "sentence", "title", "page_type"]
    missing = [c for c in required_cols if c not in sent_df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}")

    work = sent_df.copy().sort_values(["doc_id", "sentence_id"]).reset_index(drop=True)

    context_rows = []

    for doc_id, g in work.groupby("doc_id", sort=False):
        g = g.sort_values("sentence_id").reset_index(drop=True)
        sents = g["sentence"].fillna("").astype(str).tolist()

        for i, row in g.iterrows():
            start = max(0, i - window)
            end = min(len(g), i + window + 1)

            prev_sentence = sents[i - 1] if i - 1 >= 0 else ""
            next_sentence = sents[i + 1] if i + 1 < len(g) else ""
            local_context = " ".join([x for x in sents[start:end] if x]).strip()

            context_rows.append({
                "doc_id": row["doc_id"],
                "sentence_id": row["sentence_id"],
                "title": row["title"],
                "page_type": row["page_type"],
                "prev_sentence": prev_sentence,
                "hit_sentence": row["sentence"],
                "next_sentence": next_sentence,
                "local_context": local_context,
            })

    context_df = pd.DataFrame(context_rows)
    return context_df

def add_context_to_hits(hit_df, sent_df, window=1):
    """
    기존 hit_df(doc_id, sentence_id 기반)에
    prev_sentence / next_sentence / local_context / title / page_type 부착
    """
    context_df = build_sentence_context_table(sent_df, window=window)

    merged = hit_df.merge(
        context_df,
        on=["doc_id", "sentence_id", "title", "page_type"],
        how="left"
    )

    ordered_cols = [
        "doc_id", "sentence_id",
        "university", "department",
        "title", "page_type",
        "prev_sentence", "hit_sentence", "next_sentence", "local_context"
    ]
    existing = [c for c in ordered_cols if c in merged.columns]
    others = [c for c in merged.columns if c not in existing]

    return merged[existing + others]

def save_edge_qa_with_context(
    edge_df,
    sent_df,
    scope,
    out_dir,
    edge_type="concept",   # "concept" or "raw"
    top_n=10,
    sample_n=10,
    context_window=1
):
    """
    edge_df 상위 edge에 대해 hit sentence + local context QA csv 저장
    """
    ensure_dir(out_dir)

    if edge_df is None or edge_df.empty:
        print(f"[{scope}] {edge_type}: edge_df 비어 있음")
        return

    key_col = "concept_matches" if edge_type == "concept" else "raw_aliases"
    work_df = sent_df.copy()

    for rank, row in edge_df.head(top_n).iterrows():
        a = row["anchor"]
        b = row["related"]

        mask = work_df[key_col].apply(
            lambda x: isinstance(x, (list, tuple, set)) and (a in x) and (b in x)
        )

        hit_df = work_df.loc[
            mask,
            ["doc_id", "sentence_id", "university", "department", "title", "page_type", "sentence"]
        ].copy()

        if hit_df.empty:
            print(f"[건너뜀] {scope} {edge_type} {a}__{b} : hit 없음")
            continue

        hit_df = (
            hit_df.sort_values(["doc_id", "sentence_id"])
                  .head(sample_n)
                  .reset_index(drop=True)
                  .rename(columns={"sentence": "hit_sentence"})
        )

        hit_df = add_context_to_hits(hit_df, sent_df, window=context_window)

        fname = (
            f"qa_{scope}_{edge_type}_edge_{rank+1:02d}_"
            f"{normalize_filename(a)}__{normalize_filename(b)}.csv"
        )
        save_path = os.path.join(out_dir, fname)
        hit_df.to_csv(save_path, index=False, encoding="utf-8-sig")

        print(f"[저장] {fname} | rows={len(hit_df)}")

        del hit_df
        gc.collect()


def save_open_vocab_qa_with_context(
    open_vocab_df,
    sent_df,
    scope,
    out_dir,
    top_n=10,
    sample_n=10,
    context_window=1
):
    """
    open-vocab 상위 edge에 대해 hit sentence + local context QA csv 저장
    open_vocab_df는 anchor, related 컬럼을 가진다고 가정
    """
    ensure_dir(out_dir)

    if open_vocab_df is None or open_vocab_df.empty:
        print(f"[{scope}] open-vocab: 결과 비어 있음")
        return

    work_df = sent_df.copy()

    for rank, row in open_vocab_df.head(top_n).iterrows():
        a = row["anchor"]
        b = row["related"]

        mask = work_df.apply(
            lambda r: (
                isinstance(r["concept_matches"], (list, tuple, set)) and
                isinstance(r["open_vocab"], (list, tuple, set)) and
                (a in r["concept_matches"]) and
                (b in r["open_vocab"])
            ),
            axis=1
        )

        hit_df = work_df.loc[
            mask,
            ["doc_id", "sentence_id", "university", "department", "title", "page_type", "sentence"]
        ].copy()

        if hit_df.empty:
            print(f"[건너뜀] {scope} open-vocab {a}__{b} : hit 없음")
            continue

        hit_df = (
            hit_df.sort_values(["doc_id", "sentence_id"])
                  .head(sample_n)
                  .reset_index(drop=True)
                  .rename(columns={"sentence": "hit_sentence"})
        )

        hit_df = add_context_to_hits(hit_df, sent_df, window=context_window)

        fname = (
            f"qa_{scope}_open_vocab_{rank+1:02d}_"
            f"{normalize_filename(a)}__{normalize_filename(b)}.csv"
        )
        save_path = os.path.join(out_dir, fname)
        hit_df.to_csv(save_path, index=False, encoding="utf-8-sig")

        print(f"[저장] {fname} | rows={len(hit_df)}")

        del hit_df
        gc.collect()

In [31]:
# =========================
# 13-1. 실제 실행
# TARGET_SCOPES 예:
# TARGET_SCOPES = ["main"] 또는 ["main", "all"]
# =========================
QA_OUT_DIR = os.path.join(OUTPUT_DIR, "qa_with_context")
ensure_dir(QA_OUT_DIR)

TARGET_SCOPES = ["main", "all"]

for scope in TARGET_SCOPES:
    sent_df = sentence_tables[scope]

    # ESG-only concept edge
    save_edge_qa_with_context(
        edge_df=esg_only_results[scope]["concept_edges"],
        sent_df=sent_df,
        scope=scope,
        out_dir=QA_OUT_DIR,
        edge_type="concept",
        top_n=10,
        sample_n=10,
        context_window=1   # 앞뒤 1문장
    )

    # ESG-only raw edge
    save_edge_qa_with_context(
        edge_df=esg_only_results[scope]["raw_edges"],
        sent_df=sent_df,
        scope=scope,
        out_dir=QA_OUT_DIR,
        edge_type="raw",
        top_n=10,
        sample_n=10,
        context_window=1
    )

    # open-vocab edge
    save_open_vocab_qa_with_context(
        open_vocab_df=open_vocab_results[scope]["topk"],
        sent_df=sent_df,
        scope=scope,
        out_dir=QA_OUT_DIR,
        top_n=10,
        sample_n=10,
        context_window=1
    )

print("QA with context 저장 완료")

[저장] qa_main_concept_edge_01_소비__소비자.csv | rows=10
[저장] qa_main_concept_edge_02_안전__위생.csv | rows=10
[저장] qa_main_concept_edge_03_신뢰__행동.csv | rows=10
[저장] qa_main_concept_edge_04_건강__지역사회.csv | rows=10
[저장] qa_main_concept_edge_05_지속가능__지속가능성.csv | rows=10
[저장] qa_main_concept_edge_06_ESG__신뢰.csv | rows=7
[저장] qa_main_concept_edge_07_성평등__인권.csv | rows=3
[저장] qa_main_concept_edge_08_재활용__플라스틱.csv | rows=3
[저장] qa_main_concept_edge_09_개발__교육.csv | rows=10
[저장] qa_main_concept_edge_10_개발__건강.csv | rows=10
[저장] qa_main_raw_edge_01_식량__안보.csv | rows=10
[저장] qa_main_raw_edge_02_consumer__소비자.csv | rows=10
[저장] qa_main_raw_edge_03_behavior__consumption.csv | rows=10
[저장] qa_main_raw_edge_04_behavior__consumer.csv | rows=10
[저장] qa_main_raw_edge_05_consumption__women.csv | rows=10
[저장] qa_main_raw_edge_06_소비__소비자.csv | rows=10
[저장] qa_main_raw_edge_07_carbon__energy.csv | rows=8
[저장] qa_main_raw_edge_08_food__health.csv | rows=10
[저장] qa_main_raw_edge_09_안전__위생.csv | rows=10
[저장] qa_main_raw


## 셀프 점검 체크포인트

1. `sentence_level_*.csv`에서 `concept_matches`가 비어 있지 않은 문장이 실제 ESG 문장인지 20건 정도 육안 확인  
2. `esg_concept_edges` 상위 결과가 특정 한 대학에만 몰려 있지 않은지 `university_n` 확인  
3. `open_vocab_fp`에 `교육, 연구, 학생` 같은 일반어가 과도하게 상위에 남으면 stopword를 추가  
4. `raw`와 `concept`를 합쳐 단일 순위표로 해석하지 않기  
5. 결과 문장은 반드시 "함께 서술되는 경향", "공출현 경향"으로 표현하기
